# Partial Freezing of a Multilingual Model for Chinese PoS Tagging

This notebook compares full fine-tuning with a partial-freezing setup on a small Universal Dependencies Chinese GSD sample. It reports word-level UPOS accuracy on the same held-out development sample. Run every cell from top to bottom. The first run downloads the model weights and UD files.

## 1. Install the packages

Run this cell once in a fresh environment. A CUDA runtime is recommended; CPU also works but may be slow.

In [ ]:
%pip install -r requirements.txt

## 2. Check the runtime

This prints the Python, PyTorch, and device details so the run environment is recorded.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Run the controlled comparison

The script downloads and parses the UD train/dev files, samples a fixed subset, fine-tunes a fresh model for each condition, and writes measured results to `results.json`. For the frozen condition, the embeddings and first three of six DistilBERT layers stay fixed; the upper three layers and classification head train.

In [ ]:
%run run_experiment.py --max-train 600 --max-dev 150 --epochs 3 --batch-size 16 --learning-rate 2e-5 --seed 42

## 4. Read the measured scores

The values below come from the completed run. If the experiment failed earlier, fix that issue and rerun before interpreting results.

In [ ]:
import json
from pathlib import Path
results = json.loads(Path('results.json').read_text(encoding='utf-8'))
for row in results['results']:
    print(f"{row['condition']}: {row['dev_word_accuracy']:.4%} ({row['correct_words']}/{row['dev_words']} words), trainable parameters {row['trainable_parameters']:,}/{row['total_parameters']:,}")
difference = results['results'][1]['dev_word_accuracy'] - results['results'][0]['dev_word_accuracy']
print(f'Partial freezing minus full fine-tuning: {difference:+.4%}')

## 5. Interpretation notes

Compare the measured scores, training losses, and number of trainable parameters. A difference on this small sample may reflect sampling noise; this one-seed experiment does not establish a general advantage. Explain whether the result fits the idea that keeping lower multilingual representations fixed can help or limit adaptation. For a stronger claim, repeat with multiple seeds and languages.